# Deepfake Detection Model Training

This notebook trains a Deepfake Detection Model using XceptionNet (CNN) and LSTM (RNN).

## 1. Setup and Imports

In [ ]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
import matplotlib.pyplot as plt
import glob
from sklearn.model_selection import train_test_split

# Custom modules from your project
import model_utils
import preprocess

# Check GPU
gpus = tf.config.list_physical_devices('GPU')
print(f"TensorFlow Version: {tf.__version__}")
print(f"Num GPUs Available: {len(gpus)}")
if len(gpus) > 0:
    print(f"Using GPU: {gpus[0]}")
else:
    print("WARNING: Running on CPU. This will be slow.")

## 2. Configuration

In [ ]:
# Config
BATCH_SIZE = 4
EPOCHS = 20
FRAME_COUNT = 20
IMG_SIZE = (299, 299)
TEST_SPLIT = 0.2

# Ensure models directory exists
if not os.path.exists('models'):
    os.makedirs('models')

## 3. Data Generator
We use a custom Data Generator to load video frames efficiently.

In [ ]:
class VideoDataGenerator(tf.keras.utils.Sequence):
    """
    Custom Keras Data Generator to load video data in batches.
    """
    def __init__(self, file_paths, labels, batch_size=4, frame_count=20, img_size=(299, 299), shuffle=True):
        self.file_paths = file_paths
        self.labels = labels
        self.batch_size = batch_size
        self.frame_count = frame_count
        self.img_size = img_size
        self.shuffle = shuffle
        self.indexes = np.arange(len(self.file_paths))
        self.on_epoch_end()

    def __len__(self):
        return int(np.floor(len(self.file_paths) / self.batch_size))

    def __getitem__(self, index):
        # Generate indexes of the batch
        indexes = self.indexes[index*self.batch_size:(index+1)*self.batch_size]
        
        # Find list of IDs
        batch_paths = [self.file_paths[k] for k in indexes]
        batch_labels = [self.labels[k] for k in indexes]

        return self.__data_generation(batch_paths, batch_labels)

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indexes)

    def __data_generation(self, batch_paths, batch_labels):
        # Initialization
        X = np.empty((self.batch_size, self.frame_count, *self.img_size, 3))
        y = np.empty((self.batch_size), dtype=int)

        for i, path in enumerate(batch_paths):
            # Load and preprocess the video
            try:
                video_tensor = preprocess.preprocess_video(path, max_frames=self.frame_count, img_size=self.img_size)
                X[i,] = video_tensor[0]
                y[i] = batch_labels[i]
            except Exception as e:
                print(f"Error processing {path}: {e}")
                # Return zeros if error (should be handled better in proper pipeline)
                X[i,] = np.zeros((self.frame_count, *self.img_size, 3))
                y[i] = batch_labels[i]

        return X, y

## 4. Load Data Paths

In [ ]:
def load_data_paths():
    # Assuming script is run from 'backend/' directory
    # Adjust this path if 'original' is in the parent folder or elsewhere
    root_dir = os.path.dirname(os.getcwd()) # Go up one level from backend
    
    # If running notebook from backend folder directly, root_dir might need to be:
    if os.path.basename(os.getcwd()) == 'backend':
        root_dir = os.path.dirname(os.getcwd())
    else:
        # Assume we are in root
        root_dir = os.getcwd()

    real_dir = os.path.join(root_dir, "original")
    fake_dir1 = os.path.join(root_dir, "Deepfakes")
    fake_dir2 = os.path.join(root_dir, "Face2Face")
    
    print(f"Search Root: {root_dir}")
    print(f"Looking for real videos in: {real_dir}")

    real_videos = glob.glob(os.path.join(real_dir, "**", "*.mp4"), recursive=True) + \
                  glob.glob(os.path.join(real_dir, "**", "*.avi"), recursive=True)
    
    fake_videos = glob.glob(os.path.join(fake_dir1, "**", "*.mp4"), recursive=True) + \
                  glob.glob(os.path.join(fake_dir1, "**", "*.avi"), recursive=True) + \
                  glob.glob(os.path.join(fake_dir2, "**", "*.mp4"), recursive=True) + \
                  glob.glob(os.path.join(fake_dir2, "**", "*.avi"), recursive=True)
    
    print(f"Found {len(real_videos)} Real videos")
    print(f"Found {len(fake_videos)} Fake videos")
    
    X = real_videos + fake_videos
    y = [0] * len(real_videos) + [1] * len(fake_videos)
    
    return np.array(X), np.array(y)

X, y = load_data_paths()

## 5. Prepare Training and Validation Sets

In [ ]:
if len(X) > 0:
    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=TEST_SPLIT, random_state=42)

    train_gen = VideoDataGenerator(X_train, y_train, BATCH_SIZE, FRAME_COUNT, IMG_SIZE)
    val_gen = VideoDataGenerator(X_val, y_val, BATCH_SIZE, FRAME_COUNT, IMG_SIZE)
    print("Generators created.")
else:
    print("No data found! Check paths.")

## 6. Build Model

In [ ]:
model = model_utils.build_model(frame_count=FRAME_COUNT, img_size=IMG_SIZE)
model.summary()

## 7. Train Model

In [ ]:
callbacks = [
    ModelCheckpoint('model_weights.h5', save_best_only=True, monitor='val_loss', mode='min'),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=0.00001, verbose=1),
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1)
]

print("Starting training...")
history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS,
    callbacks=callbacks
)

model.save('deepfake_model_final.h5')
print("Training Complete.")

## 8. Plot Results

In [ ]:
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']

epochs_range = range(len(acc))

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label='Training Accuracy')
plt.plot(epochs_range, val_acc, label='Validation Accuracy')
plt.legend(loc='lower right')
plt.title('Training and Validation Accuracy')

plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label='Training Loss')
plt.plot(epochs_range, val_loss, label='Validation Loss')
plt.legend(loc='upper right')
plt.title('Training and Validation Loss')

plt.show()